### Ablation study on vision models

These experiements are to understand the spatial reasoning capabilities and pitfalls as demonstrated in the "What’s “up” with vision-language models? Investigating their struggle with spatial reasoning" paper but with the current sota models

### Part-1 : testing the succesor models

In [19]:
!git clone https://github.com/ksk-17/whatsup_vlms.git

Cloning into 'whatsup_vlms'...
remote: Enumerating objects: 76, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 76 (delta 13), reused 15 (delta 8), pack-reused 51 (from 1)
Receiving objects: 100% (76/76), 3.64 MiB | 24.39 MiB/s, done.
Resolving deltas: 100% (20/20), done.


In [21]:
%cd whatsup_vlms
!mkdir -p data outputs

/content/whatsup_vlms/whatsup_vlms/whatsup_vlms


In [9]:
!pip install -r requirements.txt
!pip install -q -U accelerate sentencepiece
!pip install -q -U git+https://github.com/huggingface/transformers.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [10]:
from transformers import Blip2ForImageTextRetrieval, MetaClip2Model, Siglip2Model
print("all imports OK")

all imports OK


In [11]:
# smoke test
!python main_aro.py --model-name="siglip2:google/siglip2-so400m-patch14-384" \
    --dataset=Controlled_Images_A --download --batch-size=8 --num_workers=2

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 888/888 [00:00<00:00, 2366.85it/s]
Image directory for Controlled Images A could not be found!
Downloading...
From (original): https://drive.google.com/uc?id=19KGYVQjrV3syb00GgcavB2nZTW5NXX0H
From (redirected): https://drive.google.com/uc?id=19KGYVQjrV3syb00GgcavB2nZTW5NXX0H&confirm=t&uuid=9ad35a44-af58-4c5a-bb12-a869423432e1
To: /content/whatsup_vlms/whatsup_vlms/data/controlled_images.tar.gz
100% 95.1M/95.1M [00:03<00:00, 31.2MB/s]
controlled_images/
controlled_images/water-filter_on_chair.jpeg
controlled_images/dragonfruit_on_chair.jpeg
controlled_images/headphones_unde

In [14]:
models = [
    "siglip2:google/siglip2-so400m-patch14-384",
    "metaclip2:facebook/metaclip-2-worldwide-huge-quickgelu",
    "blip2-itm:Salesforce/blip2-itm-vit-g",
]
datasets = ["Controlled_Images_A", "Controlled_Images_B", "COCO_QA_one_obj",
            "COCO_QA_two_obj", "VG_QA_one_obj", "VG_QA_two_obj"]

for model in models:
    for dataset in datasets:
        print(f"\n=== {model} on {dataset} ===")
        !python -u main_aro.py --model-name="{model}" --dataset="{dataset}" --download --batch-size=32 --num_workers=4


=== siglip2:google/siglip2-so400m-patch14-384 on Controlled_Images_A ===
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 888/888 [00:00<00:00, 2572.95it/s]
Computing retrieval scores: 100% 13/13 [00:04<00:00,  2.70it/s]
Individual accuracy: 18.689320388349515
Pair accuracy: 4.854368932038835
Set accuracy: 0.0
Saving results to ./outputs/Controlled_Images_A.csv

=== siglip2:google/siglip2-so400m-patch14-384 on Controlled_Images_B ===
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model c

In [18]:
import pandas as pd
import glob

files = glob.glob("outputs/*.csv")
all_df = pd.concat([pd.read_csv(f, index_col=0) for f in files], ignore_index=True)

# weighted overall accuracy per (Model, Dataset)
overall = (
    all_df.groupby(["Model", "Dataset"])
    .apply(lambda g: (g["Accuracy"] * g["Count"]).sum() / g["Count"].sum() * 100)
    .reset_index(name="Accuracy")
)

# roll the 6 raw datasets up into the paper's 3 groupings
group_map = {
    "Controlled_Images_A": "What'sUp", "Controlled_Images_B": "What'sUp",
    "COCO_QA_one_obj": "COCO-spatial", "COCO_QA_two_obj": "COCO-spatial",
    "VG_QA_one_obj": "GQA-spatial", "VG_QA_two_obj": "GQA-spatial",
}
overall["Group"] = overall["Dataset"].map(group_map)

summary = overall.groupby(["Model", "Group"])["Accuracy"].mean().unstack()
summary = summary[["What'sUp", "COCO-spatial", "GQA-spatial"]]  # fixed column order
summary["Avg"] = summary.mean(axis=1)
summary = summary.round(1)

summary.loc["XVLM 16M (*paper)"] = [41.9, 65.0, 58.2, 55.0]

print(summary.to_string())

Group                                                   What'sUp  COCO-spatial  GQA-spatial   Avg
Model                                                                                            
blip2-itm:Salesforce/blip2-itm-vit-g                        40.1          53.3         49.9  47.8
metaclip2:facebook/metaclip-2-worldwide-huge-quickgelu      29.4          47.1         47.0  41.2
siglip2:google/siglip2-so400m-patch14-384                   25.0          51.0         49.0  41.7
XVLM 16M (*paper)                                           41.9          65.0         58.2  55.0


/tmp/ipykernel_655/2376839193.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: (g["Accuracy"] * g["Count"]).sum() / g["Count"].sum() * 100)


Part - 2

In [22]:
for model in ["llava-onevision", "qwen2.5-vl"]:
    for dataset in ["Controlled_Images_A", "Controlled_Images_B"]:
        print(f"\n=== {model} on {dataset} ===")
        !python -u vqa_eval.py --model-name="{model}" --dataset="{dataset}"


=== llava-onevision on Controlled_Images_A ===
config.json: 100% 2.53k/2.53k [00:00<00:00, 6.23MB/s]
model.safetensors.index.json: 100% 78.0k/78.0k [00:00<00:00, 111MB/s]
Fetching 4 files: 100% 4/4 [00:43<00:00, 10.97s/it]
Download complete: 100% 16.1G/16.1G [00:43<00:00, 366MB/s]
Loading weights: 100% 765/765 [00:03<00:00, 199.05it/s]
generation_config.json: 100% 126/126 [00:00<00:00, 621kB/s]
processor_config.json: 100% 178/178 [00:00<00:00, 871kB/s]
chat_template.json: 100% 826/826 [00:00<00:00, 4.10MB/s]
preprocessor_config.json: 100% 1.73k/1.73k [00:00<00:00, 6.89MB/s]
tokenizer_config.json: 100% 1.80k/1.80k [00:00<00:00, 5.99MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 107MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 114MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 144MB/s]
added_tokens.json: 100% 122/122 [00:00<00:00, 574kB/s]
special_tokens_map.json: 100% 367/367 [00:00<00:00, 1.90MB/s]
video_preprocessor_config.json: 100% 621/621 [00:00<00:00, 3.37MB/s]
Image di

In [24]:
import pandas as pd
import glob

files = glob.glob("outputs_vqa/vqa_*.csv")
all_df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

# Accuracy per Model x Dataset
summary = (
    all_df.groupby(["Model", "Dataset"])["Correct"]
    .mean()
    .mul(100)
    .round(1)
    .unstack()
)
summary["Avg"] = summary.mean(axis=1).round(1)

print(summary.to_string())

Dataset          Controlled_Images_A  Controlled_Images_B   Avg
Model                                                          
llava-onevision                 96.8                 93.6  95.2
qwen2.5-vl                      93.0                 99.0  96.0
